In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import lora_transfer_pruning
from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map="cuda:3",
    # cache_dir="/glazkov-dev/.cache",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [5]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

https://github.com/VainF/Torch-Pruning?tab=readme-ov-file#sparse-training-optional

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [6]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [7]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [8]:
#model.eval() - dont use when train and build dep graph in torch_pruning
# out = model(evaluation_blocks[:4, :].to("cuda:3"))
# torch.cuda.empty_cache()
# print(out.logits.shape)
# print("out in GB:", out.logits.numel() * 8 / 1024 / 1024 / 1024)

In [8]:
bridge.blocks[0].attn.q

LinearBridge(4096 -> 4096, bias=False, original_component=Linear)

In [10]:
any(p.requires_grad for p in bridge.model.parameters())

True

In [11]:
model.train()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): EmbeddingBridge(
      (hook_in): HookPoint(name='embed.hook_in')
      (hook_out): HookPoint(name='embed.hook_out')
      (_original_component): Embedding(128256, 4096)
    )
    (layers): ModuleList(
      (0): BlockBridge(
        (hook_in): HookPoint(name='blocks.0.hook_in')
        (hook_out): HookPoint(name='blocks.0.hook_out')
        (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
        (_original_component): LlamaDecoderLayer(
          (self_attn): PositionEmbeddingsAttentionBridge(
            (hook_in): HookPoint(name='blocks.0.attn.hook_in')
            (hook_out): HookPoint(name='blocks.0.attn.hook_out')
            (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
            (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
            (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
            (hook_result): HookPoint(name='blocks.0.attn.hook_resu

We don't want norms like LlamaRMSNorm with x/std(x)*weight being pruned.

In [9]:
ignored_params = []
for name, param in model.named_parameters():
    if "norm" in name:
        ignored_params.append(param)

In [10]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to("cuda:3")

def trace_forward(model, input_ids):
    return model(
        input_ids=input_ids,
        #use_cache=False,
        #return_dict=True,
    ).logits #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
    unwrapped_parameters=list(zip(ignored_params, [0] * len(ignored_params)))
)


In [11]:
print("if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.")

if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.


~~unwrapped_parameters - parameters that is not in registered model.parameters()~~
unwrapped_parameters - parameters that just nn.Parameter like weight = nn.Parameter(torch.ones(5)) instead of nn.Linear()

!Torch pruning doesnt understand semantics of channels/rows of Parameter


Note: parameters setted via self.linear = nn.Linear() through `__setattr__`  
it should be nn.Parameter()

In [15]:
# 2. To prune the output channels of model.conv1, we need to find the corresponding group with a pruning function and pruning indices.
group = DG.get_pruning_group(bridge.blocks[0].attn.o._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )


In [16]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.o_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.o_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.o_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _ElementWiseOp_1470(AddBackward0), len(idxs)=3
[2] prune_out_channels on _ElementWiseOp_1470(AddBackward0) => prune_out_channels on model.embed_tokens._original_component (Embedding(128256, 4096)), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1470(AddBackward0) => prune_out_channels on _ElementWiseOp_1440(AddBackward0), len(idxs)=3
[4

For the o proj it's too much group size.  
For the q - it will be bigger.

o projection in 0 layer affect on all tensor shapes in next modules (mlp blocks, q, k, v, o, up, down, silu and so on)

Can afford to find pruning groups only on primitive modules like nn.Linear that directly participate in computational graph, not LlamaMLP or composite layers.

In [17]:
#group = DG.get_pruning_group(bridge.blocks[0].mlp.._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )
#will error
group = DG.get_pruning_group(bridge.blocks[0].mlp.up_proj._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )


In [18]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)) => prune_out_channels on model.layers.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)) => prune_out_channels on _ElementWiseOp_1474(MulBackward0), len(idxs)=3
[2] prune_out_channels on _ElementWiseOp_1474(MulBackward0) => prune_out_channels on _ElementWiseOp_1475(SiluBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1474(MulBackward0) => prune_out_channels on _Reshape_1472(), len(idxs)=3
[4] prune_out_channels on _Reshape_1472() => prune_out_channel

But pruning in_channels doesn't have fanout effect

In [19]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_in_channels, idxs=[2, 6, 9] )


In [20]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_in_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_in_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_in_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _ElementWiseOp_1555(MmBackward0), len(idxs)=3
[2] prune_out_channels on _ElementWiseOp_1555(MmBackward0) => prune_out_channels on _Reshape_1556(), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1555(MmBackward0) => prune_out_channels on _ElementWiseOp_1557(TBackward0), len(idxs)=3
[4] prune_out_channels on _Reshape_1556() => prune_out_chan

#hung ups

In [24]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )


KeyboardInterrupt: 

In [26]:
import faulthandler

trace_file = open("torch_pruning_trace_for_attn_q_dep_graph_building.txt", "w")
faulthandler.enable(file=trace_file)
faulthandler.dump_traceback_later(30, repeat=True, file=trace_file)

group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component,
    tp.prune_linear_out_channels,
    idxs=[2, 6, 9],
)

KeyboardInterrupt: 

check dep graph manually

In [14]:
target = bridge.blocks[0].attn.q._original_component
root = DG.module2node[target]

visited = set()
stack = [root]
edges = 0

while stack:
    node = stack.pop()
    if node in visited:
        continue

    visited.add(node)

    for dep in node.dependencies:
        edges += 1
        if dep.target not in visited:
            stack.append(dep.target)

print("reachable nodes:", len(visited))
print("reachable edges:", edges)
print("total DG nodes:", len(DG.module2node))

reachable nodes: 3279
reachable edges: 7262
total DG nodes: 3279


In [26]:
target = bridge.blocks[0].mlp.down_proj._original_component

root = DG.module2node[target]

visited = set()
stack = [root]
edges = 0

while stack:
    node = stack.pop()
    if node in visited:
        continue

    visited.add(node)

    for dep in node.dependencies:
        edges += 1
        if dep.target not in visited:
            stack.append(dep.target)

print("reachable nodes:", len(visited))
print("reachable edges:", edges)
print("total DG nodes:", len(DG.module2node))

reachable nodes: 3279
reachable edges: 7262
total DG nodes: 3279


Test of changed faster has_pruning_op function, that work not O(n)

In [22]:
from torch_pruning.dependency.group import Group

original_has_pruning_op = Group.has_pruning_op

def fast_has_pruning_op(self, dep, idxs):
    if not hasattr(self, "_fast_seen_ops"):
        self._fast_seen_ops = {
            (
                id(old_dep.target),
                old_dep.handler,
                frozenset(old_idxs),
            )
            for old_dep, old_idxs in self._group
        }
        self._fast_checks = 0

    key = (
        id(dep.target),
        dep.handler,
        frozenset(idxs),
    )

    self._fast_checks += 1

    if self._fast_checks % 100_000 == 0:
        print(
            "checks:", self._fast_checks,
            "unique states:", len(self._fast_seen_ops),
            "raw group size:", len(self._group),
            flush=True,
        )

    if key in self._fast_seen_ops:
        return True

    # Сразу резервируем состояние: вызывающий код после False
    # добавит его в group и processing_stack.
    self._fast_seen_ops.add(key)
    return False

Group.has_pruning_op = fast_has_pruning_op

In [23]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )

checks: 100000 unique states: 49987 raw group size: 55052
checks: 200000 unique states: 99987 raw group size: 110052
checks: 300000 unique states: 149987 raw group size: 165052
checks: 400000 unique states: 199987 raw group size: 220052
checks: 500000 unique states: 249987 raw group size: 275052
checks: 600000 unique states: 299987 raw group size: 330052
checks: 700000 unique states: 349987 raw group size: 385052
checks: 800000 unique states: 399987 raw group size: 440052
checks: 900000 unique states: 449987 raw group size: 495052
checks: 1000000 unique states: 499987 raw group size: 550052
checks: 1100000 unique states: 549987 raw group size: 605052
checks: 1200000 unique states: 599987 raw group size: 660052
checks: 1300000 unique states: 649987 raw group size: 715052
checks: 1400000 unique states: 699987 raw group size: 770052
checks: 1500000 unique states: 749987 raw group size: 825052
checks: 1600000 unique states: 799987 raw group size: 880052
checks: 1700000 unique states: 84998

KeyboardInterrupt: 

Debug and find problem edge in dep graph

In [17]:
from collections import Counter, defaultdict
from torch_pruning.dependency.group import Group

original_has_pruning_op = Group.has_pruning_op

tp_debug = {
    "seen": set(),
    "target_counts": Counter(),
    "handler_counts": Counter(),
    "max_idxs": {},
    "samples": defaultdict(list),
}

MAX_STATES = 100_000


def debug_has_pruning_op(self, dep, idxs):
    target_id = id(dep.target)
    handler_name = getattr(
        dep.handler,
        "__qualname__",
        repr(dep.handler),
    )

    canonical_idxs = frozenset(
        (x.idx, x.root_idx)
        if hasattr(x, "idx")
        else x
        for x in idxs
    )

    key = (
        target_id,
        handler_name,
        canonical_idxs,
    )

    if key in tp_debug["seen"]:
        return True

    tp_debug["seen"].add(key)
    tp_debug["target_counts"][target_id] += 1
    tp_debug["handler_counts"][handler_name] += 1

    old_max = tp_debug["max_idxs"].get(target_id, 0)
    if len(canonical_idxs) > old_max:
        tp_debug["max_idxs"][target_id] = len(canonical_idxs)

    if len(tp_debug["samples"][target_id]) < 5:
        tp_debug["samples"][target_id].append(
            list(canonical_idxs)[:20]
        )

    if len(tp_debug["seen"]) >= MAX_STATES:
        raise RuntimeError(
            f"Stopped after {MAX_STATES} unique dependency states"
        )

    return False


Group.has_pruning_op = debug_has_pruning_op

In [18]:
try:
    group = DG.get_pruning_group(
        bridge.blocks[0].attn.q._original_component, 
        tp.prune_linear_out_channels, 
        idxs=[2, 6, 9] )
finally:
    Group.has_pruning_op = original_has_pruning_op

RuntimeError: Stopped after 100000 unique dependency states

In [ ]:
tp_debug['max_idxs'] #all indices the same, shape doesnt change

{137607492398160: 3,
 137607492398112: 3,
 137607492397968: 3,
 137607492397296: 3,
 137607492397104: 3,
 137607492396960: 3,
 137607492396816: 3,
 137607492393744: 3,
 137607492393552: 3,
 137607492393408: 3,
 137607492393264: 3,
 137607492393120: 3,
 137607492392976: 3,
 137607492392832: 3,
 137607492392688: 3,
 137607492392544: 3,
 137607492389424: 3,
 137607492389232: 3,
 137607492389088: 3,
 137607492388944: 3,
 137607492388800: 3,
 137607492388656: 3,
 137607492388416: 3,
 137607492388224: 3,
 137607492388080: 3,
 137607492389520: 3,
 137607492389664: 3,
 137607492389808: 3,
 137607492389952: 3,
 137607492390240: 3,
 137607492390384: 3,
 137607492390528: 3,
 137607492390672: 3,
 137607492393840: 3,
 137607492393984: 3,
 137607492394128: 3,
 137607492394272: 3,
 137607492394416: 3,
 137607492394560: 3,
 137607492394704: 3,
 137607492394848: 3,
 137607492394992: 3,
 137607492395280: 3,
 137607492395424: 3,
 137607492395616: 3,
 137607492396672: 3,
 137607492395856: 3,
 137607492395

In [34]:
node_by_id = {
    id(node): node
    for node in DG.module2node.values()
}

for target_id, count in tp_debug["target_counts"].most_common(30):
    node = node_by_id.get(target_id)

    print(
        "\ncount:", count,
        "max idxs:", tp_debug["max_idxs"].get(target_id),
        "\nnode:", node,
        "\nnode outputs:", node.outputs,
        "\nnode module:", node.module,
        "\nsamples:", tp_debug["samples"][target_id],
    )


count: 9997 max idxs: 3 
node: <Node: (_ConcatOp_1536([0, 1024, 2048]))> 
node outputs: [<Node: (_ElementWiseOp_1535(MulBackward0))>] 
node module: _ConcatOp_1536([0, 1024, 2048]) 
samples: [[(9, None), (2, None), (6, None)], [(1030, None), (1033, None), (1026, None)], [(2054, None), (2057, None), (2050, None)], [(3074, None), (3078, None), (3081, None)], [(4098, None), (4102, None), (4105, None)]]

count: 9996 max idxs: 3 
node: <Node: (_ElementWiseOp_1531(UnsqueezeBackward0))> 
node outputs: [<Node: (_ExpandOp_1530())>] 
node module: _ElementWiseOp_1531(UnsqueezeBackward0) 
samples: [[(9, None), (2, None), (6, None)], [(1030, None), (1033, None), (1026, None)], [(2054, None), (2057, None), (2050, None)], [(3074, None), (3078, None), (3081, None)], [(4098, None), (4102, None), (4105, None)]]

count: 9996 max idxs: 3 
node: <Node: (_ConcatOp_1532(None))> 
node outputs: [<Node: (_ElementWiseOp_1531(UnsqueezeBackward0))>] 
node module: _ConcatOp_1532(None) 
samples: [[(9, None), (2, Non

In [38]:
node_by_id.get(137607492395424).inputs

[<Node: (_ElementWiseOp_1537(NegBackward0))>, <Node: (_Slice_1538())>]

In [39]:
node_by_id.get(137607492395424).outputs

[<Node: (_ElementWiseOp_1535(MulBackward0))>]

In [40]:
node_by_id.get(137607492395424).module

_ConcatOp_1536([0, 1024, 2048])

In [41]:
node_by_id.get(137607492395424).dependencies

[prune_out_channels on _ConcatOp_1536([0, 1024, 2048]) => prune_out_channels on _ElementWiseOp_1537(NegBackward0),
 prune_out_channels on _ConcatOp_1536([0, 1024, 2048]) => prune_out_channels on _Slice_1538(),
 prune_out_channels on _ConcatOp_1536([0, 1024, 2048]) => prune_out_channels on _ElementWiseOp_1535(MulBackward0)]

In [43]:
node_by_id.get(137607492395424)._name

In [44]:
node_by_id.get(137607492395424).module_class

torch_pruning.ops._ConcatOp

In [45]:
concat_node = node_by_id.get(137607492395424)

In [49]:
from collections import deque
import torch.nn as nn

queue = deque([(concat_node, 0)])
visited = set()

while queue:
    node, depth = queue.popleft()

    if node in visited or depth > 30:
        continue
    visited.add(node)

    module = node.module
    name = DG._module2name.get(module)
    print(module)

    if isinstance(module, nn.Linear):
        print(
            "UPSTREAM LINEAR:",
            name,
            module,
            "depth:",
            depth,
        )
        continue

    for input_node in node.inputs:
        queue.append((input_node, depth + 1))

_ConcatOp_1536([0, 1024, 2048])
_ElementWiseOp_1537(NegBackward0)
_Slice_1538()
_Slice_1544()
_ElementWiseOp_1539(TransposeBackward0)
_Reshape_1540()
Linear(in_features=4096, out_features=1024, bias=False)
UPSTREAM LINEAR: model.layers.0._original_component.self_attn._original_component.k_proj._original_component Linear(in_features=4096, out_features=1024, bias=False) depth: 4


In [50]:
print(
    concat_node.grad_fn,
    getattr(concat_node.grad_fn, "_saved_dim", "missing"),
)

<CatBackward0 object at 0x7d273db3e050> 18446744073709551615


In [58]:
concat_node.grad_fn.__class__.__module__

'builtins'

In [59]:
dir(concat_node.grad_fn)

['__call__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '_input_metadata',
 '_register_hook_dict',
 '_saved_dim',
 '_sequence_nr',
 '_set_sequence_nr',
 'metadata',
 'name',
 'next_functions',
 'register_hook',
 'register_prehook',
 'requires_grad']

What is 18446744073709551615? It's -1 in uint64. Great bug!

It means, that _saved_dim saved correctly, but in inner torch_pruning checks it doesn't pass check in place cat_node.grad_fn._saved_dim < constants.MAX_VALID_DIM. where constants.MAX_VALID_DIM = 100.

In [64]:
type(concat_node.grad_fn._saved_dim)

int

In [100]:
x = torch.randn(2, 3, 4, 5, requires_grad=True)

y = torch.cat(
    (x[..., :2], x[..., 2:]),
    dim=-1,
)

print(y.grad_fn)
print(y.grad_fn._saved_dim)
# print(y.grad_fn._saved_sizes)


18446744073709551615


In [101]:
#print all _saved fields
for name in sorted(dir(y.grad_fn)):
    if not name.startswith("_saved"):
        continue

    try:
        value = getattr(y.grad_fn, name)
    except Exception as exc:
        value = f"<error: {exc!r}>"

    print(name, "=", value)

_saved_dim = 18446744073709551615


In [102]:
y = torch.split(x, 2, dim=-1)[2] #2 + 2 + 1 last shapes
print(y.shape)
print(y.grad_fn)
print(y.grad_fn._saved_dim)
print(y.grad_fn._saved_self_sym_sizes)

torch.Size([2, 3, 4, 1])
18446744073709551615
(2, 3, 4, 5)


In [103]:
type(y.grad_fn._saved_dim)

int

In [112]:
1 << 1

2

In [ ]:



from torch_pruning.dependency.constants import MAX_VALID_DIM
print(MAX_VALID_DIM)

18446744073709551616


In [66]:
grad_fn = concat_node.grad_fn

print("type:", type(grad_fn))
print("name:", grad_fn.name())
print("next_functions:", grad_fn.next_functions)
print("metadata:", grad_fn.metadata)

type: <class 'CatBackward0'>
name: CatBackward0
next_functions: ((<NegBackward0 object at 0x7d273db3e140>, 0), (<SliceBackward0 object at 0x7d273db3e1d0>, 0))
metadata: {}


Original component is k!

In [47]:
queue

deque([])

In [23]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.5.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [ ]:
bridge.blocks[0].attn.q._original_component

Linear(in_features=4096, out_features=4096, bias=False)

In [29]:
bridge.blocks[0].attn.k._original_component

Linear(in_features=4096, out_features=1024, bias=False)

In [30]:
bridge.blocks[0].attn.v._original_component

Linear(in_features=4096, out_features=1024, bias=False)

That means, that concat operation - about k or v _ConcatOp_1536([0, 1024, 2048])

In [ ]:
model.config.num_attention_heads * model.config.head_dim #out size of q

4096

In [ ]:
print(group)

In [ ]:
tp.pruner.BasePruner()

In [ ]:
# 3. Do the pruning
if DG.check_pruning_group(group): # avoid over-pruning, i.e., channels=0.
    group.prune()
# 4. Save & Load
# model.zero_grad() # clear gradients to avoid a large file size
# torch.save(model, 'model.pth') # !! no .state_dict here since the structure has been changed after pruning
# model = torch.load('model.pth') # load the pruned model. you may need torch.load('model.pth', weights_only=False) for PyTorch 2.6.0+.


In [ ]:
        for i in range(self.pruning_iterations):
            potential_groups_to_prune = self.pruner.step(interactive=True)
            for group in potential_groups_to_prune:
                dep, idxs = group[0]
                layer = dep.layer
                pruning_fn = dep.pruning_fn
                pruning_hist.append((layer, idxs, pruning_fn))
                group.prune()